<table style="width:100%; border:2px solid #2583d8; border-radius:10px; background-color:white;">
<tr>

<td style="width:34%; vertical-align:middle; padding:20px;">
<img src="./assets/RoboDedito.png"
     alt="Robot character"
     style="width:100%; max-width:480px; display:block; margin:auto; border-radius:10px;">
</td>

<td style="width:66%; vertical-align:middle; padding:25px 35px;">

<h2 style="color:#1474c4; font-size:30px;">
Personal Assistant.
</h2>

<p style="font-size:20px;"> Create a new folder for this project.</p>

<p style="font-size:20px;"> Open the folder using <code>VSCode<code>.</p>

<p style="font-size:20px;"> The uv package was already installed on the local computer (see previous lab)</p>

<p style="font-size:20px;"> Create a a virtual environment. Run the command <code>uv init</code> to start <code>uv<code></p>

<p style="font-size:20px;"> Create another folde named <code>me</code></p>

<p style="font-size:20px;"> Inside this folder, place a file called <code>Profile.pdf</code> containing your LinkedIn information. Place another file named <code>summary.txt</code> containing details about your persona. </p>

</td>
</tr>
</table>

<div style="margin-top:20px; border:2px solid #2583d8; border-radius:10px; background-color:#ffffff; padding:25px 30px; font-family:Arial, sans-serif;">

<p style="font-size:28px;"> AI API Keys.</p>
<p style="font-size:20px;"> For this lab, we will need two AI API keys. One key from OpenAI and another from Google</p>
<p style="font-size:20px;"> Create another folder named <code>.env</code> and place those keys inside.
<p style="font-size:20px;"> <em>OPENAI_API_KEY=sk-proj-l6QOo</em> and <em>GOOGLE_API_KEY=AQ.Ab8R</em> </p>
</div>

<div style="margin-top:20px; border:2px solid #2583d8; border-radius:10px; background-color:#ffffff; padding:25px 30px; font-family:Arial, sans-serif;">

<p style="font-size:28px;"> Add the required libraries</p>
<p style="font-size:20px;"> uv add gradio</p>
<p style="font-size:20px;"> uv add dotenv</p>
<p style="font-size:20px;"> uv add openai</p>
<p style="font-size:20px;"> uv add pypdf</p>

</div>

<div style="margin-top:20px; border:2px solid #2583d8; border-radius:10px; background-color:#ffffff; padding:25px 30px; font-family:Arial, sans-serif;">

<p style="font-size:28px;"> Clean Personal Assistant Python Script.</p>
<p style="font-size:20px;"> The following script is the code that you can copy and paste to run your Personal Assistant:</p>

</div>

In [ ]:
import os
from pathlib import Path
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from pypdf import PdfReader

BASE_DIR = Path(__file__).resolve().parent
load_dotenv(BASE_DIR / ".env", override=True)

openai = OpenAI()

google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise RuntimeError("GOOGLE_API_KEY is not set in the .env file.")

gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",)

reader = PdfReader(BASE_DIR / "me" / "Profile.pdf")
linkedin = "\n".join(page.extract_text() or "" for page in reader.pages)

with (BASE_DIR / "me" / "summary.txt").open(encoding="utf-8") as file:
    summary = file.read()

name = "Rogelio"

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

personal_agent_prompt = (
    f"You are acting as {name}'s personal assistant. "
    f"You are answering questions related to {name}'s website, particularly questions "
    f"related to {name}'s career, background, skills, and experience. "
    f"Your responsibility is to act as {name}'s personal assistant on the website as faithfully "
    f"as possible. You are given a summary of {name}'s background and LinkedIn "
    "profile that you can use to answer questions. Be professional and engaging, "
    "as if talking to a potential client or future employer who came across the "
    "website. If you don't know the answer, say so."
    f"\n\n## Summary:\n{summary}"
    f"\n\n## LinkedIn Profile:\n{linkedin}\n\n"
    f"With this context, chat with the user while always staying in character as {name}'s personal assistant.")

evaluator_agent_prompt = (
    "You are an evaluator that decides whether a response to a question is "
    "acceptable. You are provided with a conversation between a User and an "
    "Agent. Decide whether the Agent's latest response is of acceptable quality. "
    f"Here is the agent's role description: {personal_agent_prompt}\n\n"
    "Evaluate the latest response and report whether it is acceptable, along "
    "with your feedback.")

def evaluator_user_prompt(reply, message, history):
    return (
        f"Here is the conversation between the User and the Agent:\n\n{history}\n\n"
        f"Here is the latest message from the User:\n\n{message}\n\n"
        f"Here is the latest response from the Agent:\n\n{reply}\n\n"
        "Evaluate the response and report whether it is acceptable, along with "
        "your feedback.")

def evaluate(reply, message, history):
    messages = [
        {"role": "system", "content": evaluator_agent_prompt},
        {
            "role": "user",
            "content": evaluator_user_prompt(reply, message, history),
        },]
    response = gemini.beta.chat.completions.parse(
        model="gemini-3.5-flash",
        messages=messages,
        response_format=Evaluation,)
    return response.choices[0].message.parsed

def rerun(reply, message, history, feedback):
    updated_personal_agent_prompt = (
        personal_agent_prompt
        + "\n\n## Previous answer rejected\n"
        "Your previous reply was rejected by quality control.\n"
        f"## Your attempted answer:\n{reply}\n\n"
        f"## Reason for rejection:\n{feedback}\n\n")
    messages = [
        {"role": "system", "content": updated_personal_agent_prompt},
        *history,
        {"role": "user", "content": message},]
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,)
    return response.choices[0].message.content

def chat(message, history):
    messages = [
        {"role": "system", "content": personal_agent_prompt},
        *history,
        {"role": "user", "content": message},]
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,)
    reply = response.choices[0].message.content
    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
        return reply

    print("Failed evaluation - retrying")
    print(evaluation.feedback)
    return rerun(reply, message, history, evaluation.feedback)

def main():
    welcome_message = [
        {
            "role": "assistant",
            "content": f"Hello! I am {name}'s personal assistant. Do you have questions about him? Feel free to ask!"
        }
    ]
    chatbot = gr.Chatbot(value=welcome_message)
    gr.ChatInterface(fn=chat, chatbot=chatbot).launch()

if __name__ == "__main__":
    main()


<div style="margin-top:20px; border:2px solid #2583d8; border-radius:10px; background-color:#ffffff; padding:25px 30px; font-family:Arial, sans-serif;">

<p style="font-size:28px;"> How do the previous code work?</p>
<p style="font-size:20px;"> The following is the same script with comments added, providing an explanation:</p>

</div>

In [ ]:
import os  # Lets Python access environment variables, such as API keys.
from pathlib import Path  # Provides an easy way to work with file and folder paths.
import gradio as gr  # Imports Gradio for creating the web-based chat interface.
from dotenv import load_dotenv  # Loads environment variables from a .env file.
from openai import OpenAI  # Imports the OpenAI client.
from pydantic import BaseModel  # Imports BaseModel for defining structured data models.
from pypdf import PdfReader  # Imports PdfReader so the program can read PDF files.


BASE_DIR = Path(__file__).resolve().parent  # Gets the folder where this Python script is located.
load_dotenv(BASE_DIR / ".env", override=True)  # Loads variables from the .env file in the project folder.


openai = OpenAI()  # Creates an OpenAI client using the OPENAI_API_KEY environment variable.


google_api_key = os.getenv("GOOGLE_API_KEY")  # Reads the Google API key from the environment.
if not google_api_key:  # Checks whether the Google API key was found.
    raise RuntimeError("GOOGLE_API_KEY is not set in the .env file.")  # Stops the program if the key is missing.


gemini = OpenAI(  # Creates a second OpenAI-compatible client for Google Gemini.
    api_key=google_api_key,  # Supplies the Google API key to the Gemini client.
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",)  # Uses Google's OpenAI-compatible API endpoint.


reader = PdfReader(BASE_DIR / "me" / "Profile.pdf")  # Opens the Profile.pdf file.
linkedin = "\n".join(page.extract_text() or "" for page in reader.pages)  # Extracts and combines text from all PDF pages.


with (BASE_DIR / "me" / "summary.txt").open(encoding="utf-8") as file:  # Opens summary.txt for reading.
    summary = file.read()  # Reads the entire contents of summary.txt into a variable.


name = "Rogelio"  # Stores the name that the personal assistant will represent.


class Evaluation(BaseModel):  # Defines the structure expected from the evaluator model.
    is_acceptable: bool  # Stores whether the answer passed the evaluation.
    feedback: str  # Stores comments explaining the evaluator's decision.


personal_agent_prompt = (  # Starts building the system prompt for the personal assistant.
    f"You are acting as {name}'s personal assistant. "  # Defines the assistant's main role.
    f"You are answering questions related to {name}'s website, particularly questions "  # Limits questions to website-related topics.
    f"related to {name}'s career, background, skills, and experience. "  # Specifies the areas the assistant should discuss.
    f"Your responsibility is to act as {name}'s personal assistant on the website as faithfully "  # Tells the assistant to represent the person accurately.
    f"as possible. You are given a summary of {name}'s background and LinkedIn "  # Explains that profile information will be provided.
    "profile that you can use to answer questions. Be professional and engaging, "  # Sets the assistant's tone.
    "as if talking to a potential client or future employer who came across the "  # Describes the expected audience.
    "website. If you don't know the answer, say so."  # Prevents the assistant from inventing unknown information.
    f"\n\n## Summary:\n{summary}"  # Adds the summary.txt content to the prompt.
    f"\n\n## LinkedIn Profile:\n{linkedin}\n\n"  # Adds the extracted PDF profile information.
    f"With this context, chat with the user while always staying in character as {name}'s personal assistant.")  # Reinforces the assistant's role.


evaluator_agent_prompt = (  # Starts building the system prompt for the evaluator agent.
    "You are an evaluator that decides whether a response to a question is "  # Defines the evaluator's role.
    "acceptable. You are provided with a conversation between a User and an "  # Explains what information the evaluator receives.
    "Agent. Decide whether the Agent's latest response is of acceptable quality. "  # Tells Gemini to judge the response quality.
    f"Here is the agent's role description: {personal_agent_prompt}\n\n"  # Gives the evaluator the personal assistant's instructions.
    "Evaluate the latest response and report whether it is acceptable, along "  # Requests an accept/reject decision.
    "with your feedback.")  # Requests feedback explaining the decision.


def evaluator_user_prompt(reply, message, history):  # Defines a function that creates the evaluator's user prompt.
    return (  # Returns the completed evaluator prompt.
        f"Here is the conversation between the User and the Agent:\n\n{history}\n\n"  # Adds previous conversation history.
        f"Here is the latest message from the User:\n\n{message}\n\n"  # Adds the user's newest message.
        f"Here is the latest response from the Agent:\n\n{reply}\n\n"  # Adds the assistant's newest answer.
        "Evaluate the response and report whether it is acceptable, along with "  # Tells the evaluator what decision to make.
        "your feedback.")  # Requests feedback along with the decision.


def evaluate(reply, message, history):  # Defines a function that evaluates an AI-generated reply.
    messages = [  # Creates the message list that will be sent to Gemini.
        {"role": "system", "content": evaluator_agent_prompt},  # Sends the evaluator's instructions.
        {  # Starts the evaluator's user message.
            "role": "user",  # Identifies this message as coming from the user.
            "content": evaluator_user_prompt(reply, message, history),  # Builds the evaluation request.
        },]  # Ends the list of messages.

    response = gemini.beta.chat.completions.parse(  # Sends the evaluation request to Gemini.
        model="gemini-3.5-flash",  # Selects the Gemini model used for evaluation.
        messages=messages,  # Sends the evaluator conversation.
        response_format=Evaluation,)  # Requires Gemini to return data matching the Evaluation model.

    return response.choices[0].message.parsed  # Returns Gemini's structured evaluation result.


def rerun(reply, message, history, feedback):  # Defines a function that retries an answer if evaluation fails.
    updated_personal_agent_prompt = (  # Creates an updated prompt containing the evaluator's feedback.
        personal_agent_prompt  # Starts with the original personal assistant instructions.
        + "\n\n## Previous answer rejected\n"  # Adds a section explaining that the previous answer failed.
        "Your previous reply was rejected by quality control.\n"  # Tells the assistant that the previous answer was rejected.
        f"## Your attempted answer:\n{reply}\n\n"  # Includes the previous answer.
        f"## Reason for rejection:\n{feedback}\n\n")  # Includes the evaluator's explanation.

    messages = [  # Creates a new list of messages for the retry.
        {"role": "system", "content": updated_personal_agent_prompt},  # Sends the revised system prompt.
        *history,  # Adds all previous conversation messages.
        {"role": "user", "content": message},]  # Adds the user's current question again.

    response = openai.chat.completions.create(  # Sends the retry request to OpenAI.
        model="gpt-4o-mini",  # Selects the OpenAI model.
        messages=messages,)  # Sends the complete conversation.

    return response.choices[0].message.content  # Returns the improved response from OpenAI.


def chat(message, history):  # Defines the main function Gradio will call when the user sends a message.
    messages = [  # Creates the conversation that will be sent to OpenAI.
        {"role": "system", "content": personal_agent_prompt},  # Gives OpenAI the personal assistant instructions.
        *history,  # Includes previous conversation messages.
        {"role": "user", "content": message},]  # Adds the user's newest message.

    response = openai.chat.completions.create(  # Sends the conversation to OpenAI.
        model="gpt-4o-mini",  # Selects the OpenAI model used by the personal assistant.
        messages=messages,)  # Sends all conversation messages.

    reply = response.choices[0].message.content  # Extracts the assistant's text response.
    evaluation = evaluate(reply, message, history)  # Sends that response to Gemini for quality evaluation.


    if evaluation.is_acceptable:  # Checks whether Gemini approved the response.
        print("Passed evaluation - returning reply")  # Prints a success message in the terminal.
        return reply  # Sends the approved reply back to the user.


    print("Failed evaluation - retrying")  # Prints a message if Gemini rejects the answer.
    print(evaluation.feedback)  # Prints Gemini's reason for rejecting the answer.
    return rerun(reply, message, history, evaluation.feedback)  # Generates a new answer using the feedback.


def main():  # Defines the main function that starts the application.
    welcome_message = [  # Creates the initial message displayed in the chatbot.
        {  # Starts the assistant message dictionary.
            "role": "assistant",  # Identifies the welcome message as coming from the assistant.
            "content": f"Hello! I am {name}'s personal assistant. Do you have questions about him? Feel free to ask!"  # Defines the welcome text.
        }  # Ends the assistant message dictionary.
    ]  # Ends the welcome message list.

    chatbot = gr.Chatbot(value=welcome_message)  # Creates a Gradio chatbot containing the welcome message.
    gr.ChatInterface(fn=chat, chatbot=chatbot).launch()  # Creates and launches the web chat interface.


if __name__ == "__main__":  # Checks whether this file is being run directly.
    main()  # Starts the application.

<div style="margin-top:20px; border:2px solid #2583d8; border-radius:10px; background-color:#ffffff; padding:25px 30px; font-family:Arial, sans-serif;">

<p style="font-size:28px;"> Using open-source llms</p>
<p style="font-size:20px;"> Instead of using Google and OpenAI, we can use open-source models like Falcon and Wizard-Vicuna as follows:</p>

</div>

In [ ]:
from pathlib import Path
import gradio as gr
from openai import OpenAI
from pydantic import BaseModel
from pypdf import PdfReader

# Get the folder where this Python script is located.
BASE_DIR = Path(__file__).resolve().parent

# Connect to the local Ollama server.
# "ollama" is only a placeholder API key and is not a real secret key.
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

# Open the PDF containing profile information.
reader = PdfReader(BASE_DIR / "me" / "Profile.pdf")

# Extract the text from all pages of the PDF.
linkedin = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

# Open and read the summary file.
with (BASE_DIR / "me" / "summary.txt").open(encoding="utf-8") as file:
    summary = file.read()

# Name of the person represented by the assistant.
name = "Rogelio"

# Defines the structured result expected from the evaluator.
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

# System prompt for the main personal assistant.
personal_agent_prompt = (
    f"You are acting as {name}'s personal assistant. "
    f"You are answering questions related to {name}'s website, particularly questions "
    f"related to {name}'s career, background, skills, and experience. "
    f"Your responsibility is to act as {name}'s personal assistant on the website as faithfully "
    f"as possible. You are given a summary of {name}'s background and LinkedIn "
    "profile that you can use to answer questions. Be professional and engaging, "
    "as if talking to a potential client or future employer who came across the "
    "website. If you don't know the answer, say so."
    f"\n\n## Summary:\n{summary}"
    f"\n\n## LinkedIn Profile:\n{linkedin}\n\n"
    f"With this context, chat with the user while always staying in character "
    f"as {name}'s personal assistant."
)

# System prompt for Falcon, which acts as the evaluator.
evaluator_agent_prompt = (
    "You are an evaluator that decides whether a response to a question is "
    "acceptable. You are provided with a conversation between a User and an "
    "Agent. Decide whether the Agent's latest response is of acceptable quality. "
    f"Here is the agent's role description: {personal_agent_prompt}\n\n"
    "Evaluate the latest response and report whether it is acceptable, "
    "along with your feedback."
)

# Builds the prompt sent to the evaluator.
def evaluator_user_prompt(reply, message, history):
    return (
        f"Here is the conversation between the User and the Agent:\n\n"
        f"{history}\n\n"
        f"Here is the latest message from the User:\n\n"
        f"{message}\n\n"
        f"Here is the latest response from the Agent:\n\n"
        f"{reply}\n\n"
        "Evaluate the response and report whether it is acceptable, "
        "along with your feedback."
    )

# Uses Falcon to evaluate the response produced by Wizard Vicuna.
def evaluate(reply, message, history):
    messages = [
        {
            "role": "system",
            "content": evaluator_agent_prompt,
        },
        {
            "role": "user",
            "content": evaluator_user_prompt(
                reply,
                message,
                history,
            ),
        },
    ]

    response = ollama.beta.chat.completions.parse(
        model="falcon:40b",
        messages=messages,
        response_format=Evaluation,
    )
    return response.choices[0].message.parsed

# Generates another response when Falcon rejects the first answer.
def rerun(reply, message, history, feedback):

    updated_personal_agent_prompt = (
        personal_agent_prompt
        + "\n\n## Previous answer rejected\n"
        + "Your previous reply was rejected by quality control.\n"
        + f"## Your attempted answer:\n{reply}\n\n"
        + f"## Reason for rejection:\n{feedback}\n\n"
    )

    messages = [
        {
            "role": "system",
            "content": updated_personal_agent_prompt,
        },
        *history,
        {
            "role": "user",
            "content": message,
        },
    ]

    response = ollama.chat.completions.create(
        model="wizard-vicuna-uncensored:30b",
        messages=messages,
    )

    return response.choices[0].message.content

# Main chat function used by Gradio.
def chat(message, history):
    messages = [
        {
            "role": "system",
            "content": personal_agent_prompt,
        },
        *history,
        {
            "role": "user",
            "content": message,
        },
    ]

    # Wizard Vicuna generates the first answer.
    response = ollama.chat.completions.create(
        model="wizard-vicuna-uncensored:30b",
        messages=messages,
    )

    reply = response.choices[0].message.content

    # Falcon evaluates Wizard Vicuna's answer.
    evaluation = evaluate(
        reply,
        message,
        history,
    )

    # Return the answer if Falcon approves it.
    if evaluation.is_acceptable:

        print("Passed evaluation - returning reply")

        return reply

    # Otherwise display the evaluator feedback.
    print("Failed evaluation - retrying")
    print(evaluation.feedback)

    # Ask Wizard Vicuna to generate another answer.
    return rerun(
        reply,
        message,
        history,
        evaluation.feedback,
    )

# Starts the Gradio application.
def main():
    welcome_message = [
        {
            "role": "assistant",
            "content": (
                f"Hello! I am {name}'s personal assistant. "
                "Do you have questions about him? Feel free to ask!"
            ),
        }
    ]

    chatbot = gr.Chatbot(
        value=welcome_message
    )

    gr.ChatInterface(
        fn=chat,
        chatbot=chatbot,
    ).launch()

# Run the application only when this file is executed directly.
if __name__ == "__main__":
    main()

<div style="margin-top:20px; border:2px solid #2583d8; border-radius:10px; background-color:#ffffff; padding:25px 30px; font-family:Arial, sans-serif;">

<p style="font-size:28px;"> Merging these two ideas together</p>
<p style="font-size:20px;"> Instead of using two diferent scripts, we can use LiteLLm to orchestrate the diferent types of LLMs models.</p>
<p style="font-size:20px;"> Here is the script that merges those two scripts:</p>

</div>

In [ ]:
import os
from pathlib import Path
import gradio as gr
from dotenv import load_dotenv
from litellm import completion
from pydantic import BaseModel
from pypdf import PdfReader

# ---------------------------------------------------------
# 1. PROJECT SETUP
# ---------------------------------------------------------

# Gets the folder where this Python script is located.
BASE_DIR = Path(__file__).resolve().parent

# Loads environment variables from the .env file.
load_dotenv(BASE_DIR / ".env", override=True)

# ---------------------------------------------------------
# 2. API KEYS *** REMOVE THIS ENTIRE SECTION IF YOU PLAN TO USE FREE MODELS
# ---------------------------------------------------------

# Reads the OpenAI API key from the environment.
openai_api_key = os.getenv("OPENAI_API_KEY")

# Stops the program if the OpenAI API key is missing.
if not openai_api_key:
    raise RuntimeError("OPENAI_API_KEY is not set in the .env file.")

# Reads the Google API key from the environment.
google_api_key = os.getenv("GEMINI_API_KEY")

# Stops the program if the Google API key is missing.
if not google_api_key:
    raise RuntimeError("GEMINI_API_KEY is not set in the .env file.")

# LiteLLM reads provider API keys from environment variables.
os.environ["OPENAI_API_KEY"] = openai_api_key.strip()
os.environ["GEMINI_API_KEY"] = google_api_key.strip()

# ---------------------------------------------------------
# 3a. UNCOMMNET THIS WHOLE SECTION TO USE PAID MODEL CONFIGURATION
# ---------------------------------------------------------

# Main model used by the personal assistant.
MAIN_MODEL = "openai/gpt-4o-mini"

# Gemini model used to evaluate the assistant's answers.
EVALUATOR_MODEL = "gemini/gemini-2.5-flash"

# ---------------------------------------------------------
# 3b. UNCOMMENT THIS SECTION TO USE FREE MODEL CONFIGURATION
# ---------------------------------------------------------

# Main model used by the personal assistant.
#MAIN_MODEL = "ollama/wizard-vicuna-uncensored:30b"

# Falcon model used to evaluate the assistant's answers.
#EVALUATOR_MODEL = "ollama/falcon:40b"

# ---------------------------------------------------------
# 4. LOAD PROFILE INFORMATION
# ---------------------------------------------------------

# Opens the PDF containing profile information.
reader = PdfReader(BASE_DIR / "me" / "Profile.pdf")

# Extracts and combines the text from every PDF page.
linkedin = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

# Opens the summary.txt file.
with (BASE_DIR / "me" / "summary.txt").open(
    encoding="utf-8"
) as file:
    # Reads the entire summary file.
    summary = file.read()

# Name of the person represented by the assistant.
name = "Rogelio"

# ---------------------------------------------------------
# 5. STRUCTURED EVALUATION RESULT
# ---------------------------------------------------------

# Defines the structure returned by the evaluator.
class Evaluation(BaseModel):
    # True if the answer is acceptable.
    is_acceptable: bool

    # Contains feedback explaining the evaluator's decision.
    feedback: str

# ---------------------------------------------------------
# 6. PERSONAL ASSISTANT SYSTEM PROMPT
# ---------------------------------------------------------

personal_agent_prompt = (
    f"You are acting as {name}'s personal assistant. "
    f"You are answering questions related to {name}'s website, particularly questions "
    f"related to {name}'s career, background, skills, and experience. "
    f"Your responsibility is to act as {name}'s personal assistant on the website as faithfully "
    f"as possible. You are given a summary of {name}'s background and LinkedIn "
    "profile that you can use to answer questions. Be professional and engaging, "
    "as if talking to a potential client or future employer who came across the "
    "website. If you don't know the answer, say so."
    f"\n\n## Summary:\n{summary}"
    f"\n\n## LinkedIn Profile:\n{linkedin}\n\n"
    f"With this context, chat with the user while always staying in character "
    f"as {name}'s personal assistant."
)

# ---------------------------------------------------------
# 7. EVALUATOR SYSTEM PROMPT
# ---------------------------------------------------------
evaluator_agent_prompt = (
    "You are an evaluator that decides whether a response to a question is "
    "acceptable. You are provided with a conversation between a User and an "
    "Agent. Decide whether the Agent's latest response is of acceptable quality. "
    f"Here is the agent's role description: {personal_agent_prompt}\n\n"
    "Evaluate the latest response and report whether it is acceptable, "
    "along with your feedback."
)

# ---------------------------------------------------------
# 8. CREATE THE EVALUATOR USER PROMPT
# ---------------------------------------------------------
def evaluator_user_prompt(reply, message, history):
    return (
        f"Here is the conversation between the User and the Agent:\n\n"
        f"{history}\n\n"
        f"Here is the latest message from the User:\n\n"
        f"{message}\n\n"
        f"Here is the latest response from the Agent:\n\n"
        f"{reply}\n\n"
        "Evaluate the response and report whether it is acceptable, "
        "along with your feedback."
    )

# ---------------------------------------------------------
# 9. EVALUATE THE MAIN AGENT'S RESPONSE
# ---------------------------------------------------------
def evaluate(reply, message, history):
    # Creates the messages sent to Gemini.
    messages = [
        {
            "role": "system",
            "content": evaluator_agent_prompt,
        },

        {
            "role": "user",
            "content": evaluator_user_prompt(
                reply,
                message,
                history,
            ),
        },
    ]

    # Sends the evaluation request to Gemini through LiteLLM.
    response = completion(
        model=EVALUATOR_MODEL,
        messages=messages,
        response_format=Evaluation,
    )

    # Gets the structured JSON-like response from Gemini.
    result = response.choices[0].message.content

    # Converts the returned JSON text into an Evaluation object.
    evaluation = Evaluation.model_validate_json(result)

    return evaluation

# ---------------------------------------------------------
# 10. RETRY A REJECTED ANSWER
# ---------------------------------------------------------
def rerun(reply, message, history, feedback):
    # Adds the evaluator's feedback to the original system prompt.
    updated_personal_agent_prompt = (
        personal_agent_prompt
        + "\n\n## Previous answer rejected\n"
        + "Your previous reply was rejected by quality control.\n"
        + f"## Your attempted answer:\n{reply}\n\n"
        + f"## Reason for rejection:\n{feedback}\n\n"
    )

    # Rebuilds the conversation.
    messages = [
        {
            "role": "system",
            "content": updated_personal_agent_prompt,
        },
        *history,
        {
            "role": "user",
            "content": message,
        },
    ]

    # Sends the retry request to OpenAI through LiteLLM.
    response = completion(
        model=MAIN_MODEL,
        messages=messages,
    )

    # Returns the improved answer.
    return response.choices[0].message.content

# ---------------------------------------------------------
# 11. MAIN CHAT FUNCTION
# ---------------------------------------------------------
def chat(message, history):
    # Creates the conversation sent to the main model.
    messages = [
        {
            "role": "system",
            "content": personal_agent_prompt,
        },

        *history,

        {
            "role": "user",
            "content": message,
        },
    ]


    # Sends the conversation to OpenAI through LiteLLM.
    response = completion(
        model=MAIN_MODEL,
        messages=messages,
    )

    # Extracts the model's answer.
    reply = response.choices[0].message.content

    # Sends the answer to Gemini for evaluation.
    evaluation = evaluate(
        reply,
        message,
        history,
    )

    # Checks whether Gemini approved the answer.
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
        return reply

    # If Gemini rejected the answer, display the feedback.
    print("Failed evaluation - retrying")
    print(evaluation.feedback)

    # Generates another answer using the evaluator's feedback.
    return rerun(
        reply,
        message,
        history,
        evaluation.feedback,
    )

# ---------------------------------------------------------
# 12. START THE GRADIO APPLICATION
# ---------------------------------------------------------
def main():
    # Initial greeting displayed in the chatbot.
    welcome_message = [
        {
            "role": "assistant",
            "content": (
                f"Hello! I am {name}'s personal assistant. "
                "Do you have questions about him? Feel free to ask!"
            ),
        }
    ]

    # Creates the chatbot.
    chatbot = gr.Chatbot(
        value=welcome_message
    )

    # Creates and launches the Gradio interface.
    gr.ChatInterface(
        fn=chat,
        chatbot=chatbot,
    ).launch()

# ---------------------------------------------------------
# 13. RUN THE APPLICATION
# ---------------------------------------------------------
# Runs main() only when this file is executed directly.
if __name__ == "__main__":
    main()

<table style="width:100%; border:2px solid #2583d8; border-radius:10px; background-color:white;">
<tr>

<td style="width:34%; vertical-align:middle; padding:20px;">
<img src="assets/RoboShowi.png"
     alt="Robot showing"
     style="width:100%; max-width:480px; display:block; margin:auto; border-radius:10px;">
</td>

<td style="width:66%; vertical-align:middle; padding:25px 35px;">

<h2 style="color:#1474c4; font-size:30px;">
That's it !!!
</h2>

<p style="font-size:20px;"> Now you have:</p>

<p style="font-size:20px;"> Your own virtual assistant.</p>

<p style="font-size:20px;"> You can use paid llms or.</p>

<p style="font-size:20px;"> Open-souce models. </p>

<p style="font-size:20px;"> Everything in a Python LLM application using SDK. </p>

</td>
</tr>
</table>